# Filter-and-refine spatial join: connect two geometry layers

This lab joins a collection of points to a collection of polygons. We will inspect bounding-box candidates, apply the exact predicate, compare the result with a brute-force join and see how boundary rules change the answer.

**You need:** Python and Shapely 2.1+.

The index is built over the right-hand polygons. The returned pairs always use the order (left input index, right tree index).

In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (current, *current.parents)
        if (candidate / "implementations/python/spatial-join").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Run this notebook from a checkout of geo-dev-hub/spatial-analysis."
    )

implementation_dir = repo_root / "implementations/python/spatial-join"
sys.path.insert(0, str(implementation_dir.resolve()))

import matplotlib.pyplot as plt
from shapely import STRtree, box
from shapely.geometry import Point
from spatial_join import spatial_join


## 1. Prepare two geometry layers

The left side contains points. The right side contains service areas. We will use `within` to ask which points fall inside each area.

In [ ]:
points = [
    Point(0.5, 0.5),
    Point(1.5, 0.5),
    Point(2.5, 1.5),
    Point(3.0, 3.0),
    Point(5.0, 5.0),
]
areas = [
    box(0, 0, 2, 2),
    box(2, 1, 4, 3),
    box(4, 4, 6, 6),
]

tree = STRtree(areas)
print('Left geometries:', len(points))
print('Right geometries:', len(areas))
print('Possible pairs without an index:', len(points) * len(areas))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for i, area in enumerate(areas):
    x, y = area.exterior.xy
    ax.fill(x, y, alpha=0.2)
    ax.text(area.centroid.x, area.centroid.y, f'Area {i}', ha='center', va='center')
for i, point in enumerate(points):
    ax.scatter(point.x, point.y, s=70)
    ax.annotate(f'P{i}', (point.x, point.y), xytext=(5, 5), textcoords='offset points')
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Points and service-area polygons')
plt.show()


## 2. See the candidate pairs

First query the index without an exact predicate. These are bounding-box candidates. Then ask the tree to refine those candidates using the exact `within` predicate.

In [ ]:
candidate_pairs = tree.query(points)
exact_pairs = tree.query(points, predicate='within')

print('Bounding-box candidate pairs (left index, right index):')
print(candidate_pairs.T.tolist())
print('Exact within pairs:')
print(exact_pairs.T.tolist())
print('Candidate count:', candidate_pairs.shape[1])
print('Exact match count:', exact_pairs.shape[1])


## 3. Compare with the reusable implementation

The project helper returns sorted index pairs. Its results should match Shapely's exact predicate query.

In [ ]:
indexed_matches = spatial_join(points, areas, predicate='within')
print('Indexed join:', indexed_matches)
assert indexed_matches == sorted(map(tuple, exact_pairs.T.tolist()))


## 4. Boundary rules change the result

A point exactly on a polygon boundary is not within the polygon under the usual topological definition, but it does intersect it. Try the same point with both predicates.

In [ ]:
boundary_point = Point(0, 1)
print('within:', spatial_join([boundary_point], [areas[0]], predicate='within'))
print('intersects:', spatial_join([boundary_point], [areas[0]], predicate='intersects'))


## 5. Check the indexed result against brute force

For a small dataset, the simplest correctness check is to evaluate every pair directly. The indexed result should match it exactly.

In [ ]:
brute_force = [
    (i, j)
    for i, point in enumerate(points)
    for j, area in enumerate(areas)
    if point.within(area)
]
print('Brute-force pairs:', brute_force)
print('Indexed pairs:', indexed_matches)
assert indexed_matches == brute_force


## What to take away

- The index reduces the number of pairs that need exact geometry checks.
- Bounding-box candidates are not the same as final matches.
- Predicate direction matters: Shapely tests the input geometry against the tree geometry.
- Boundary rules change whether a point is within, intersects or is covered by a polygon.
- The returned indices refer to the original left and right input sequences.

Continue with the [filter-and-refine spatial join lesson](../algorithms/query-processing/filter-refine-spatial-join.md), the [STR-packed R-tree lesson](../algorithms/indexing/str-packed-rtree.md) and the [tested implementation](../implementations/python/spatial-join/README.md).

**Reference:** [Shapely STRtree query API](https://shapely.readthedocs.io/en/stable/strtree.html).